# Lawler HI Summer Experiment Calculations

In [5]:
# testing hysteresis parameter calculation? 
import numpy as np
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates


experiment_directory = 'pulse_experiments'
experiments = {}
for filename in os.listdir(experiment_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(experiment_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]  # drop stray unnamed columns
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        experiments[key] = df                                      # store the data frame in the dictionary

# calculate a shear stress column for each experiment
for experiment_name, experiment_df in experiments.items():
    # calculate shear stress using the formula: shear_stress = density * gravity * water_depth * slope
    density = 1000  # kg/m^3
    gravity = 9.81  # m/s^2
    slope = 0.098
    water_depth = experiment_df['Depth']  # in cm
    shear_stress = density * gravity * water_depth/1000 * slope
    experiments[experiment_name]['shear_stress'] = shear_stress

In [6]:
experiments['Experiment1']

,Lab ID,Depth,SSC (uL/L),SSC (mg/L),DOC (mg/L),POC (mg/L),shear_stress
Date_Time,,,,,,,
2022-08-02 11:57:09,1,9.275,97.69,167.044168,2.640,10.974524,8.916800
2022-08-02 11:57:21,2,22.750,347.61,145.078620,2.384,12.330497,21.871395
2022-08-02 11:57:32,3,19.250,191.85,76.137885,2.284,NaN,18.506565
2022-08-02 11:57:42,4,16.100,176.97,47.967685,2.423,4.967149,15.478218
2022-08-02 11:57:53,5,13.884,127.27,67.326074,2.488,8.890353,13.347800
2022-08-02 11:58:04,6,12.775,83.04,61.268422,2.473,NaN,12.281630
2022-08-02 11:58:14,7,12.134,68.69,40.782355,2.314,NaN,11.665385
2022-08-02 11:58:25,8,11.959,62.04,43.350237,2.402,NaN,11.497143
2022-08-02 11:58:34,9,11.959,61.07,37.341299,2.278,NaN,11.497143


Hysteresis index calculation functions

In [7]:
def normalize_storm_df(df, tau_col, constituent_cols, cols_to_normalize=None):
    out = df.copy()

    if cols_to_normalize is None:
        cols_to_normalize = [tau_col] + [c for c in constituent_cols if c in out.columns]

    for col in cols_to_normalize:
        if col not in out.columns:
            continue

        series = pd.to_numeric(out[col], errors="coerce")
        smin = series.min(skipna=True)
        smax = series.max(skipna=True)

        if pd.isna(smin) or pd.isna(smax) or smax == smin:
            out[col] = np.nan
        else:
            out[col] = (series - smin) / (smax - smin)

    return out

def split_hydrograph(df, tau_col, constituent_col):
    ts = df[[tau_col, constituent_col]].dropna().sort_index()
    if ts.empty:
        return None, None, None

    peak_pos = ts[tau_col].to_numpy().argmax()
    peak_label = ts.index[peak_pos]

    rise = ts.iloc[:peak_pos + 1][[tau_col, constituent_col]]
    fall = ts.iloc[peak_pos:][[tau_col, constituent_col]]  # exclude the peak point from the falling limb
    return rise, fall, peak_label

def interpolate_time_at_tau(tau_target, limb_df, tau_col, allow_extrapolate=True, fit_points=4):
    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col]].dropna().sort_index()
    if len(limb) == 0:
        return None
    if len(limb) == 1:
        return limb.index[0]

    tau = limb[tau_col].to_numpy(dtype=float)
    t = limb.index.view("int64")

    exact = np.where(tau == tau_target)[0]
    if exact.size > 0:
        return limb.index[exact[0]]

    diffs = tau - tau_target
    crossings = np.where(diffs[:-1] * diffs[1:] <= 0)[0]

    if crossings.size > 0:
        i = crossings[0]
        tau0, tau1 = tau[i], tau[i + 1]
        t0, t1 = t[i], t[i + 1]
    else:
        if not allow_extrapolate:
            return None

        n = min(fit_points, len(tau))
        if n < 2:
            return limb.index[-1]

        if tau_target < np.nanmin(tau):
            tau_fit = tau[:n]
            t_fit = t[:n]
        else:
            tau_fit = tau[-n:]
            t_fit = t[-n:]

        if np.all(tau_fit == tau_fit[0]):
            return pd.to_datetime(int(t_fit[0]))

        slope, intercept = np.polyfit(tau_fit, t_fit, 1)
        t_target = slope * tau_target + intercept
        return pd.to_datetime(int(t_target))

    if tau1 == tau0:
        return pd.to_datetime(int(t0))

    t_target = t0 + (t1 - t0) * ((tau_target - tau0) / (tau1 - tau0))
    return pd.to_datetime(int(t_target))

def get_time_bracket_points(t_target, limb_df, tau_col, constituent_col, allow_extrapolate=True, fit_points=4):
    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col, constituent_col]].dropna().sort_index()
    if len(limb) < 2:
        return None

    t = limb.index.view("int64")
    tau = limb[tau_col].to_numpy(dtype=float)
    c = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    extrapolated = False

    if t[0] <= t_target_int <= t[-1]:
        upper_idx = np.searchsorted(t, t_target_int, side="right")
        lower_idx = max(upper_idx - 1, 0)
        upper_idx = min(upper_idx, len(t) - 1)
    elif not allow_extrapolate:
        return None
    else:
        n = min(fit_points, len(t))
        if n < 2:
            return None

        extrapolated = True
        if t_target_int < t[0]:
            lower_idx, upper_idx = 0, n - 1
        else:
            lower_idx, upper_idx = len(t) - n, len(t) - 1

    return {
        "t0": pd.to_datetime(int(t[lower_idx])),
        "t1": pd.to_datetime(int(t[upper_idx])),
        "tau0": tau[lower_idx],
        "tau1": tau[upper_idx],
        "c0": c[lower_idx],
        "c1": c[upper_idx],
        "extrapolated": extrapolated,
    }

def interpolate_constituent_at_time(t_target, limb_df, constituent_col, allow_extrapolate=True):
    if limb_df is None or limb_df.empty:
        return np.nan

    limb = limb_df[[constituent_col]].dropna().sort_index()
    if len(limb) == 0:
        return np.nan
    if len(limb) == 1:
        return limb[constituent_col].iloc[0]

    t = limb.index.view("int64")
    y = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    if t[0] <= t_target_int <= t[-1]:
        return np.interp(t_target_int, t, y)

    if not allow_extrapolate:
        return np.nan

    if t_target_int < t[0]:
        t0, t1 = t[0], t[1]
        y0, y1 = y[0], y[1]
    else:
        t0, t1 = t[-2], t[-1]
        y0, y1 = y[-2], y[-1]

    if t1 == t0:
        return y0

    slope = (y1 - y0) / (t1 - t0)
    return y0 + slope * (t_target_int - t0)

def calculate_lawler_HI(df, tau_col, constituent_col, storm_name, k=0.5):
    rise, fall, peak_label = split_hydrograph(df, tau_col, constituent_col)
    if rise is None or fall is None or rise.empty or fall.empty:
        print(f"Not enough data points to split the hydrograph for {storm_name}")
        return None

    tau_min = df[tau_col].min()
    tau_max = df[tau_col].max()
    tau_mid = k * (tau_max - tau_min) + tau_min

    t_rise_mid = interpolate_time_at_tau(tau_mid, rise, tau_col, allow_extrapolate=True)
    t_fall_mid = interpolate_time_at_tau(tau_mid, fall, tau_col, allow_extrapolate=True)

    if t_rise_mid is None or t_fall_mid is None:
        print(f"Could not find target time for storm {storm_name}")
        return None

    c_rise = interpolate_constituent_at_time(t_rise_mid, rise, constituent_col, allow_extrapolate=True)
    c_fall = interpolate_constituent_at_time(t_fall_mid, fall, constituent_col, allow_extrapolate=True)

    if np.isnan(c_rise) or np.isnan(c_fall):
        print(f"Could not interpolate storm {storm_name}")
        return None

    rise_bracket = get_time_bracket_points(t_rise_mid, rise, tau_col, constituent_col)
    fall_bracket = get_time_bracket_points(t_fall_mid, fall, tau_col, constituent_col)

    if c_rise > c_fall:
        HI = 1 - (c_fall / c_rise)
    elif c_rise < c_fall:
        HI = -1 + (c_rise / c_fall)
    else:
        HI = 0

    return {
        "storm_name": storm_name,
        "HI": HI,
        "tau_mid": tau_mid,
        "c_rise": c_rise,
        "c_fall": c_fall,
        "rise": rise,
        "fall": fall,
        "tau_col": tau_col,
        "constituent_col": constituent_col,
        "k": k,
        "t_rise_mid": t_rise_mid,
        "t_fall_mid": t_fall_mid,
        "rise_bracket": rise_bracket,
        "fall_bracket": fall_bracket,
        "peak_label": peak_label,
    }

def plot_lawler_hysteresis(results, df, tau_col, constituent_col,
                           xlabel=r"Shear Stress ($\tau$)",
                           ylabel="Constituent",
                           out_dir="HI_calculations/lawler_plots",
                           save=True,
                           show=False):

    rise = results["rise"]
    fall = results["fall"]
    HI = results["HI"]
    tau_mid = results["tau_mid"]
    c_rise = results["c_rise"]
    c_fall = results["c_fall"]
    storm_name = results.get("storm_name", "")
    k = results.get("k", 0.5)
    ts_df = df[[tau_col, constituent_col]].dropna().sort_index()
    plot_time_min = ts_df.index.min()
    plot_time_max = ts_df.index.max()

    def is_valid_plot_time(t):
        if t is None or pd.isna(t):
            return False
        t = pd.Timestamp(t)
        return plot_time_min <= t <= plot_time_max

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [1.2, 1]})

    ax = axes[0]
    ax.plot(ts_df.index, ts_df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.axhline(tau_mid, linestyle="--", color="tab:blue", alpha=0.7, linewidth=1.5, label=fr"$\tau_{{mid}}$")
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(MaxNLocator(8))

    ax2 = ax.twinx()
    ax2.plot(ts_df.index, ts_df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    t_rise_mid = results.get("t_rise_mid")
    t_fall_mid = results.get("t_fall_mid")

    if is_valid_plot_time(t_rise_mid):
        ax.axvline(t_rise_mid, color="forestgreen", linestyle="--", alpha=0.5)
        ax2.scatter(t_rise_mid, c_rise, s=80, color="forestgreen", edgecolor="black", zorder=6, marker="D")

    if is_valid_plot_time(t_fall_mid):
        ax.axvline(t_fall_mid, color="firebrick", linestyle="--", alpha=0.5)
        ax2.scatter(t_fall_mid, c_fall, s=80, color="firebrick", edgecolor="black", zorder=6, marker="D")

    # plot the event peak as a black star (time-series axes and twin)
    peak_label = results.get("peak_label")
    if peak_label is not None and peak_label in ts_df.index:
        tau_peak = ts_df.at[peak_label, tau_col]
        c_peak = ts_df.at[peak_label, constituent_col]
        ax.scatter(peak_label, tau_peak, marker="*", color="k", s=160, zorder=9)

    ax.set_xlabel("Date time")
    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)

    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(plot_time_min, plot_time_max)

    ax = axes[1]
    combined = pd.concat([rise, fall.iloc[1:]])
    ax.plot(combined[tau_col], combined[constituent_col], color="black", linewidth=2, alpha=0.6, zorder=1)
    ax.scatter(rise[tau_col], rise[constituent_col], color="forestgreen", label="Rising limb", zorder=4)
    ax.scatter(fall[tau_col], fall[constituent_col], color="firebrick", label="Falling limb", zorder=4)

    ax.axvline(tau_mid, linestyle="--", color="gray", linewidth=2, label=fr"$\tau_{{mid}}$ (k={k})", zorder=2)
    ax.scatter(tau_mid, c_rise, s=80, color="forestgreen", edgecolor="black", zorder=5, marker="D")
    ax.scatter(tau_mid, c_fall, s=80, color="firebrick", edgecolor="black", zorder=5, marker="D")

    # also mark peak on tau-vs-constituent panel
    if peak_label is not None and peak_label in ts_df.index:
        ax.scatter(tau_peak, c_peak, marker="*", color="k", s=160, zorder=9, edgecolor="black")

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_title(f"Lawler HI = {HI:.2f}")

    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    fig.suptitle(main_title, fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_lawler.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)

Normalize the shear stress, groundwater and constituents in each storm

In [8]:
constituents = ["SSC (mg/L)", "DOC (mg/L)", "POC (mg/L)"]

for experiment_name, experiment_df in experiments.items():
    experiments[experiment_name] = normalize_storm_df(
        experiment_df,
        tau_col="Depth",
        constituent_cols=constituents
    )

Calculate constituents hysteresis 

In [10]:
all_results = []

for experiment_name, experiment_df in experiments.items():
    for constituent in ["SSC (mg/L)", "DOC (mg/L)", "POC (mg/L)"]:
        if constituent not in experiment_df.columns:
            continue
        result = calculate_lawler_HI(
            experiment_df,
            tau_col="Depth",
            constituent_col=constituent,
            storm_name=experiment_name)

        if result is not None:
            result["experiment"] = experiment_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('HI_calculations/lawler_experiment_hysteresis.csv', index=False)

Plot constituents

In [11]:
for experiment_name, experiment_df in experiments.items():
    for constituent in ["SSC (mg/L)", "DOC (mg/L)", "POC (mg/L)"]:
        if constituent not in experiment_df.columns:
            continue

        result = calculate_lawler_HI(
            experiment_df,
            tau_col="Depth",
            constituent_col=constituent,
            storm_name=experiment_name
        )

        if result is not None:
            plot_lawler_hysteresis(result, experiment_df, "Depth", constituent, 
                                xlabel="Normalized depth",
                                ylabel="Normalized constituent",
                                out_dir="plots/lawler")

C:\Users\nicol\AppData\Local\Temp\ipykernel_12644\4230838295.py:302: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  fig.tight_layout(rect=[0, 0, 1, 0.95])
C:\Users\nicol\AppData\Local\Temp\ipykernel_12644\4230838295.py:307: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")
C:\Users\nicol\AppData\Local\Temp\ipykernel_12644\4230838295.py:302: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  fig.tight_layout(rect=[0, 0, 1, 0.95])
C:\Users\nicol\AppData\L